# Adam Optimizer

## Load Dataset from sklearn

In [1]:
import pandas as pd # pandas 임포트
import seaborn as sns # seaborn 임포트
import matplotlib.pyplot as plt # matplotlib 임포트

from sklearn.preprocessing import StandardScaler # 스케일러 임포트

In [7]:
from sklearn.datasets import fetch_california_housing # 데이터 로더 임포트
california = fetch_california_housing() # 데이터 로드

In [8]:
df = pd.DataFrame(california.data, columns=california.feature_names) # 데이터프레임 변환
df["Target"] = california.target # 정답 컬럼 추가
df.tail() # 하단 데이터 확인

In [10]:
scaler = StandardScaler() # 스케일러 생성
scaler.fit(df.values[:, :-1]) # 통계량 계산
df.iloc[:, :-1] = scaler.transform(df.values[:, :-1]) # 표준화 적용

df.tail() # 결과 확인

## Train Model with PyTorch

In [13]:
import torch # torch 임포트
import torch.nn as nn # nn 임포트
import torch.nn.functional as F # functional 임포트
import torch.optim as optim # optim 임포트

In [14]:
data = torch.from_numpy(df.values).float() # 텐서 변환

data.shape # 형상 확인

In [15]:
x = data[:, :-1] # 입력 데이터 추출
y = data[:, -1:] # 정답 데이터 추출

print(x.shape, y.shape) # 형상 출력

In [16]:
n_epochs = 4000 # 에포크 설정
batch_size = 256 # 배치 크기 설정
print_interval = 200 # 출력 간격 설정

### Build Model

In [17]:
model = nn.Sequential( # 모델 정의
    nn.Linear(x.size(-1), 6), # 선형 레이어
    nn.LeakyReLU(), # 활성화 함수
    nn.Linear(6, 5), # 선형 레이어
    nn.LeakyReLU(), # 활성화 함수
    nn.Linear(5, 4), # 선형 레이어
    nn.LeakyReLU(), # 활성화 함수
    nn.Linear(4, 3), # 선형 레이어
    nn.LeakyReLU(), # 활성화 함수
    nn.Linear(3, y.size(-1)), # 출력 레이어
)

model # 모델 구조 확인

In [18]:
optimizer = optim.Adam(model.parameters()) # Adam 최적화기 설정

In [19]:
for i in range(n_epochs): # 에포크 반복
    indices = torch.randperm(x.size(0)) # 무작위 인덱스
    x_ = torch.index_select(x, dim=0, index=indices) # 데이터 셔플
    y_ = torch.index_select(y, dim=0, index=indices) # 라벨 셔플
    
    x_ = x_.split(batch_size, dim=0) # 배치 분할
    y_ = y_.split(batch_size, dim=0) # 배치 분할
    
    y_hat = [] # 예측값 리스트
    total_loss = 0 # 손실 초기화
    
    for x_i, y_i in zip(x_, y_): # 미니배치 루프
        y_hat_i = model(x_i) # 예측
        loss = F.mse_loss(y_hat_i, y_i) # 손실 계산

        optimizer.zero_grad() # 경사 초기화
        loss.backward() # 역전파

        optimizer.step() # 가중치 업데이트
        
        total_loss += float(loss) # 손실 합산
        y_hat += [y_hat_i] # 예측값 저장

    total_loss = total_loss / len(x_) # 평균 손실
    if (i + 1) % print_interval == 0: # 로그 출력
        print('Epoch %d: loss=%.4e' % (i + 1, total_loss))
    
y_hat = torch.cat(y_hat, dim=0) # 예측값 병합
y = torch.cat(y_, dim=0) # 실제값 병합

## Let's see the result!

In [20]:
df = pd.DataFrame(torch.cat([y, y_hat], dim=1).detach().numpy(), # 결과 병합
                  columns=["y", "y_hat"]) # 컬럼명 설정

sns.pairplot(df, height=5) # 시각화
plt.show() # 출력